The distribution `lm-format-enforcer` installs ONE importable package, `lmformatenforcer` (pure Python). It answers one question at every generation step: which characters, and so which tokens, may come next so that the output still fits a JSON schema or a regular expression. vLLM offers it as a structured-output backend. The examples drive the parsers character by character and the token enforcer with a small hand-made vocabulary: no model, no GPU and no network are needed.

## Example 1 — the import, the version and the parsers the package offers

In [ ]:
import importlib.metadata
import lmformatenforcer
from lmformatenforcer import CharacterLevelParserConfig, JsonSchemaParser, RegexParser, StringParser, TokenEnforcer

# The package has no __version__ attribute: the installed metadata is the version of record.
print("lm-format-enforcer", importlib.metadata.version("lm-format-enforcer"))
config = CharacterLevelParserConfig()
assert config.max_consecutive_whitespaces == 12 and config.max_json_array_length == 20, config
assert set("0123456789abcxyzABC{}[]:,\"") <= set(config.alphabet), config.alphabet
exact = StringParser("hello")
assert exact.get_allowed_characters() == "h" and not exact.can_end()
for character in "hello":
    exact = exact.add_character(character)
assert exact.can_end() and exact.get_allowed_characters() == ""
print("OK -- StringParser walks 'hello' one character at a time and can end only after the last one")

## Example 2 — a JSON schema: valid text is accepted, a wrong type is refused at the first bad character

In [ ]:
from pydantic import BaseModel
from lmformatenforcer import JsonSchemaParser

class Answer(BaseModel):
    city: str
    population: int

def first_refused(parser, text):
    """Feed `text`; return (final parser, None) or (None, the first character the parser does not allow)."""
    for character in text:
        if character not in parser.get_allowed_characters():
            return None, character
        parser = parser.add_character(character)
    return parser, None

parser = JsonSchemaParser(Answer.model_json_schema())
assert set(parser.get_allowed_characters()) == set("{ \t\n\r"), sorted(parser.get_allowed_characters())
done, refused = first_refused(parser, '{"city": "Paris", "population": 2100000}')
assert refused is None and done.can_end()
partial, refused = first_refused(parser, '{"city": "Paris"')
assert refused is None and not partial.can_end()  # "population" is required: the object cannot close yet
assert "}" not in partial.get_allowed_characters()
_, refused = first_refused(parser, '{"city": 5')  # a number where the schema wants a string
assert refused == "5", refused
print("OK -- the schema accepts a full answer, keeps an unfinished one open and refuses a number for `city`")

## Example 3 — regular expressions, alternatives and sequences

In [ ]:
from lmformatenforcer import RegexParser, SequenceParser, StringParser, UnionParser

def accepts(parser, text):
    for character in text:
        if character not in parser.get_allowed_characters():
            return False
        parser = parser.add_character(character)
    return parser.can_end()

phone = RegexParser(r"\d{3}-\d{4}")
assert accepts(phone, "555-1234") and not accepts(phone, "555-123") and not accepts(phone, "55a-1234")
assert set(phone.get_allowed_characters()) == set("0123456789")
after_three = phone.add_character("5").add_character("5").add_character("5")
assert after_three.get_allowed_characters() == "-", after_three.get_allowed_characters()
either = UnionParser([StringParser("yes"), StringParser("no")])
assert set(either.get_allowed_characters()) == {"y", "n"}
assert accepts(either, "yes") and accepts(either, "no") and not accepts(either, "ye")
both = SequenceParser([StringParser("id="), RegexParser(r"[0-9]+")])
assert accepts(both, "id=42") and not accepts(both, "id=") and not accepts(both, "42")
print("OK -- a phone-number regex, a yes/no choice and a prefix-then-digits sequence each accept exactly their own strings")

## Example 4 — from characters to tokens: the enforcer drives a generation step by step

In [ ]:
from lmformatenforcer import RegexParser, TokenEnforcer, TokenEnforcerTokenizerData

characters = list('0123456789-{}":, abcdefghijklmnopqrstuvwxyz')
vocabulary = [(token_id, text, False) for token_id, text in enumerate(characters)]  # one token per character
end_token = len(vocabulary)
decode = lambda token_ids: "".join(characters[t] for t in token_ids if t < end_token)
tokenizer_data = TokenEnforcerTokenizerData(vocabulary, decode, end_token, False, end_token + 1)
enforcer = TokenEnforcer(tokenizer_data, RegexParser(r"\d{3}-\d{4}"))
generated, allowed_per_step = [], []
for _ in range(20):
    allowed = sorted(enforcer.get_allowed_tokens(generated).allowed_tokens)
    allowed_per_step.append(allowed)
    generated.append(allowed[-1])  # a stand-in for the model: always take the last allowed token
    if generated[-1] == end_token:
        break
digits = sorted(characters.index(c) for c in "0123456789")
assert allowed_per_step[0] == digits and allowed_per_step[3] == [characters.index("-")], allowed_per_step[:4]
assert allowed_per_step[-1] == [end_token], allowed_per_step[-1]  # once the pattern is complete only "end" is allowed
assert decode(generated) == "999-9999", decode(generated)
print("OK -- 9 steps: digits, then '-', then digits, then only the end token; generated", decode(generated))

## Example 5 — the bundled schema model, and what a schema the parser cannot follow looks like

In [ ]:
import lmformatenforcer.external.jsonschemaobject as bundled  # a trimmed copy of datamodel-code-generator's schema model
from lmformatenforcer import JsonSchemaParser, LMFormatEnforcerException

schema = {"type": "object", "properties": {"tags": {"type": "array", "items": {"enum": ["red", "green"]}},
                                           "level": {"type": "integer", "minimum": 1}}, "required": ["tags"]}
model = bundled.JsonSchemaObject(**schema)
assert sorted(model.properties) == ["level", "tags"] and model.required == ["tags"]
parser = JsonSchemaParser(schema)
for character in '{"tags": ["red", "green"]}':
    assert character in parser.get_allowed_characters(), character
    parser = parser.add_character(character)
assert parser.can_end()
opened = JsonSchemaParser(schema)
for character in '{"tags": ["':
    opened = opened.add_character(character)
assert set(opened.get_allowed_characters()) == {"r", "g"}, opened.get_allowed_characters()  # only the enum's values
assert issubclass(LMFormatEnforcerException, Exception)
anything = JsonSchemaParser(None)  # no schema: any JSON value is allowed
assert {"{", "[", '"', "t", "f", "n", "1"} <= set(anything.get_allowed_characters())
print("OK -- an enum inside an array limits the next characters to", sorted(opened.get_allowed_characters()))